# Final Model v3 — Two-stage Host-Graph GNN → Temporal Transformer
**Intelligent Adaptive IDS using Hybrid Transformer and Graph Neural Networks — NF-CSE-CIC-IDS2018-v3**

Kaggle: **GPU T4** → add the NF-CSE-CIC-IDS2018-v3 dataset → **Run All**.

Decisions taken from the Protocol v2 results, and why:

| Decision | Why (Protocol v2 evidence) |
|---|---|
| **Final model = Two-stage**: GATv2 flow embeddings (frozen) → Temporal Transformer | Same accuracy as end-to-end Hybrid (no significant difference on any split), simpler, ~half the training time, and the GNN and Transformer can run as separate stages in real time |
| **Drop TTL / TCP-window features** (28 features instead of 32) | Removing them did not hurt (slightly better); they fingerprint the attacker's machine, not attack behaviour |
| **DropEdge = 0.15** on the GNN while training | GATv2 collapsed to chance (0.07) on a rewired test graph → too dependent on graph structure |
| **Graded rewiring test** (0–100% of test flows get a random destination) | 100% rewiring is extreme; real networks change gradually. Measures robustness with and without DropEdge |
| **10 seeds** | Two-stage vs Transformer difference (~0.02–0.04) was not significant with 5 seeds because of seed variance |
| **BENIGN_POOL = 200,000** | Val/Test Benign share was 60% instead of the 80% target |
| **Final model package** (weights + preprocessing + FAR-1% threshold), reload-verified | Step 3 of the plan: everything after this (explainability, real-time, dashboard) uses this package without retraining |

Unchanged rules: (Class × Day) sampling, train-only preprocessing, no SMOTE, class-weighted loss,
validation-only model selection, test touched once per run, results saved after every run.

## 1. Environment

In [ ]:
import sys, subprocess, importlib.util
for pkg, mod in [("torch-geometric", "torch_geometric"), ("pyarrow", "pyarrow")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, gc, re, json, time, math, random, shutil, zipfile, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, ttest_rel, binomtest

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef,
                             roc_auc_score, confusion_matrix)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
elif not os.environ.get("IDS_ALLOW_CPU"):
    raise RuntimeError("No GPU. Kaggle: Settings -> Accelerator -> GPU T4, then Run All.")

## 2. Configuration
Full run ≈ 3 splits × 10 seeds × 3 models + DropEdge control (roughly 3–4 h on a T4).
`QUICK = True` → 3 seeds, 2 splits (first check that everything runs).

In [ ]:
QUICK = False

SPLIT_SEED = 42                                   # the split is FIXED; seeds only change model training
SEEDS = list(range(42, 52))                       # 10 seeds
SPLITS_TO_RUN = ["random", "host_disjoint", "temporal"]
RUN_HYBRID = False                                # end-to-end Hybrid already evaluated in Protocol v2 (ablation)

# ---- sampling ----
MAX_PER_CLASS = 5_000
BENIGN_POOL = 200_000            # enough Benign for an 80% Benign share in Val/Test
EVAL_BENIGN_RATIO = 0.80
CHUNK_SIZE = 500_000
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---- features: locked list MINUS host-fingerprint features ----
LOCKED_FEATURES = [
    "IN_BYTES", "LONGEST_FLOW_PKT", "MAX_IP_PKT_LEN", "OUT_BYTES", "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT", "DURATION_IN", "FLOW_DURATION_MILLISECONDS", "CLIENT_TCP_FLAGS",
    "SRC_TO_DST_IAT_MAX", "SRC_TO_DST_IAT_AVG", "SRC_TO_DST_IAT_STDDEV", "TCP_FLAGS", "DURATION_OUT",
    "DST_TO_SRC_IAT_AVG", "DST_TO_SRC_IAT_MAX", "NUM_PKTS_UP_TO_128_BYTES", "OUT_PKTS",
    "DST_TO_SRC_IAT_STDDEV", "IN_PKTS", "L7_PROTO", "TCP_WIN_MAX_IN", "MAX_TTL", "MIN_TTL",
    "SERVER_TCP_FLAGS", "TCP_WIN_MAX_OUT", "NUM_PKTS_512_TO_1024_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "SRC_TO_DST_SECOND_BYTES", "SHORTEST_FLOW_PKT", "MIN_IP_PKT_LEN", "NUM_PKTS_256_TO_512_BYTES",
]
FINGERPRINT_FEATURES = ["MIN_TTL", "MAX_TTL", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT"]
MODEL_FEATURES = [f for f in LOCKED_FEATURES if f not in FINGERPRINT_FEATURES]     # 28 features
LOG1P_FEATURES = True

# ---- windows ----
K = 32
WINDOW_GROUP = "src"             # best in Protocol v2 (src > dst > global)

# ---- models ----
GNN_HIDDEN, GNN_HEADS, EMB_DIM, DROPOUT = 64, 2, 128, 0.20
D_MODEL, N_HEADS, N_LAYERS, FF_DIM = 128, 4, 2, 256

# ---- training ----
BATCH_SIZE = 512
LR = 1e-3
HYBRID_GNN_LR = 1e-4
HYBRID_FREEZE_EPOCHS = 3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.0
DROPEDGE = 0.15                  # GNN message-edge dropout while training (main runs)
MAX_EPOCHS = 150
PATIENCE = 20
MIN_EPOCHS = 10

# ---- robustness ----
REWIRE_FRACTIONS = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0]
N_ROBUST_SEEDS = 3               # robustness + no-DropEdge control on the random split for the first 3 seeds
RUN_DROPEDGE_CONTROL = True

# ---- evaluation ----
TARGET_FAR = 0.01
MERGE_GROUPS = {"FTP-BruteForce": "FTP/SlowHTTP", "DoS_attacks-SlowHTTPTest": "FTP/SlowHTTP"}
N_BOOTSTRAP = 1000
PACKAGE_SPLIT = "random"         # the final deployable model is trained on the random split, first seed

if QUICK:
    SEEDS, SPLITS_TO_RUN = SEEDS[:3], ["random", "host_disjoint"]

_e = os.environ.get               # offline smoke-test overrides (not used on Kaggle)
MAX_EPOCHS = int(_e("IDS_MAX_EPOCHS", MAX_EPOCHS)); MIN_EPOCHS = min(MIN_EPOCHS, MAX_EPOCHS)
MAX_PER_CLASS = int(_e("IDS_MAX_PER_CLASS", MAX_PER_CLASS)); BENIGN_POOL = int(_e("IDS_BENIGN_POOL", BENIGN_POOL))
if _e("IDS_SEEDS"): SEEDS = [int(s) for s in _e("IDS_SEEDS").split(",")]
if _e("IDS_K"): K = int(_e("IDS_K"))
N_ROBUST_SEEDS = min(N_ROBUST_SEEDS, len(SEEDS))

WORK_DIR = Path("/kaggle/working/FINAL_V3") if Path("/kaggle/working").exists() else Path("./FINAL_V3")
RES, FIG, MOD, PRED = (WORK_DIR / d for d in ["results", "figures", "models", "predictions"])
for p in (RES, FIG, MOD, PRED):
    p.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, bool))},
          open(RES / "run_config.json", "w"), indent=2)
print("Output:", WORK_DIR, "| splits:", SPLITS_TO_RUN, "| seeds:", SEEDS, "| features:", len(MODEL_FEATURES))

## 3. Locate NF-CSE-CIC-IDS2018-v3 (CSV or ZIP)

In [ ]:
SRC_IP, DST_IP, SRC_PORT, DST_PORT = "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "L4_SRC_PORT", "L4_DST_PORT"
T_START, T_END, PROTO, LABEL, BIN_LABEL = "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "PROTOCOL", "Attack", "Label"
REQUIRED = [T_START, SRC_IP, DST_IP, LABEL]

def _has_schema(read_head):
    try:
        cols = [str(c).strip() for c in read_head().columns]
        return all(c in cols for c in REQUIRED)
    except Exception:
        return False

def find_source():
    roots = [Path(_e("IDS_DATA_DIR"))] if _e("IDS_DATA_DIR") else [Path("/kaggle/input")]
    for root in roots:
        for p in sorted(root.rglob("*.csv")):
            if "unsw" not in p.name.lower() and _has_schema(lambda: pd.read_csv(p, nrows=0)):
                return {"type": "csv", "path": p}
        for zp in sorted(root.rglob("*.zip")):
            try:
                with zipfile.ZipFile(zp) as z:
                    for m in z.namelist():
                        if m.lower().endswith(".csv") and "unsw" not in m.lower():
                            if _has_schema(lambda: pd.read_csv(z.open(m), nrows=0)):
                                return {"type": "zip", "path": zp, "member": m}
            except Exception:
                pass
    raise FileNotFoundError("NF-CSE-CIC-IDS2018-v3 CSV not found under /kaggle/input.")

def iter_chunks(src, usecols=None):
    kw = dict(chunksize=CHUNK_SIZE, low_memory=False, on_bad_lines="skip", usecols=usecols)
    if src["type"] == "csv":
        yield from pd.read_csv(src["path"], **kw)
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            yield from pd.read_csv(f, **kw)

SOURCE = find_source()
print("Dataset:", SOURCE)

def day_of(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors="coerce"), unit="ms", utc=True).dt.strftime("%Y-%m-%d")

## 4. Pass 1 — full-file Class × Day counts
This answers the team's question directly: **on which days does each class (including Benign) exist?**

In [ ]:
counts = defaultdict(int)
t0 = time.time()
for ch in iter_chunks(SOURCE, usecols=[T_START, LABEL]):
    lab = ch[LABEL].astype(str).str.strip()
    ok = ~lab.str.lower().isin(["", "nan", "none", "attack"])
    for (c, d), n in pd.DataFrame({"c": lab[ok], "d": day_of(ch.loc[ok, T_START])}).value_counts().items():
        counts[(c, d)] += int(n)
class_day = pd.Series(counts).unstack(fill_value=0).sort_index()
class_day.to_csv(RES / "full_dataset_class_by_day.csv")
print(f"Pass 1 done in {time.time() - t0:.0f}s — rows: {class_day.values.sum():,}")
display(class_day)
print("\nDays on which Benign exists:", int((class_day.loc["Benign"] > 0).sum()) if "Benign" in class_day.index else 0,
      "of", class_day.shape[1])

fig, ax = plt.subplots(figsize=(1.0 * class_day.shape[1] + 4, 0.45 * class_day.shape[0] + 1.5))
ax.imshow(np.log10(class_day.values + 1), cmap="Blues", aspect="auto")
ax.set_xticks(range(class_day.shape[1])); ax.set_xticklabels(class_day.columns, rotation=45, ha="right")
ax.set_yticks(range(class_day.shape[0])); ax.set_yticklabels(class_day.index)
for i in range(class_day.shape[0]):
    for j in range(class_day.shape[1]):
        v = class_day.values[i, j]
        if v:
            ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=7,
                    color="white" if np.log10(v + 1) > 0.6 * np.log10(class_day.values.max() + 1) else "black")
ax.set_title("Full dataset: flows per class per day"); plt.tight_layout()
plt.savefig(FIG / "01_class_by_day.png", dpi=160, bbox_inches="tight"); plt.show()

## 5. Pass 2 — uniform sampling stratified by (Class, Day)
Quota of a (class, day) cell = cap × (rows in that cell / rows of that class). Rows inside a cell are chosen
**uniformly at random over the whole file** (random key + keep the smallest keys). Classes smaller than the cap are kept whole.

In [ ]:
cls_tot = class_day.sum(1)
cap = {c: (BENIGN_POOL if c == "Benign" else MAX_PER_CLASS) for c in class_day.index}
quota = {}
for c in class_day.index:
    for d in class_day.columns:
        n = int(class_day.loc[c, d])
        if n:
            quota[(c, d)] = n if cls_tot[c] <= cap[c] else max(1, int(cap[c] * n / cls_tot[c]))

rng_s = np.random.default_rng(SPLIT_SEED)
kept = {}
t0 = time.time()
for ch in iter_chunks(SOURCE):
    ch.columns = [str(c).strip() for c in ch.columns]
    ch[LABEL] = ch[LABEL].astype(str).str.strip()
    ch = ch[~ch[LABEL].str.lower().isin(["", "nan", "none", "attack"])]
    ch = ch.assign(_day=day_of(ch[T_START]).values, _r=rng_s.random(len(ch)))
    for key, g in ch.groupby([LABEL, "_day"], sort=False):
        q = quota.get(key, 0)
        if q == 0:
            continue
        g = g if key not in kept else pd.concat([kept[key], g], ignore_index=True)
        kept[key] = g.nsmallest(q, "_r") if len(g) > q else g
    del ch; gc.collect()
df = pd.concat(kept.values(), ignore_index=True).drop(columns="_r")
del kept; gc.collect()
print(f"Pass 2 done in {time.time() - t0:.0f}s — sampled {len(df):,} rows")

# ---- cleaning (same structural rules as before) ----
df["_ts"] = pd.to_datetime(pd.to_numeric(df[T_START], errors="coerce"), unit="ms", utc=True)
n0 = len(df)
df = df.dropna(subset=[SRC_IP, DST_IP, "_ts", LABEL])
dkey = [c for c in [T_START, T_END, SRC_IP, DST_IP, SRC_PORT, DST_PORT, PROTO, LABEL] if c in df.columns]
n_dup = int(df.duplicated(subset=dkey).sum())
df = df.drop_duplicates(subset=dkey).sort_values("_ts", kind="mergesort").reset_index(drop=True)
df[SRC_IP] = df[SRC_IP].astype(str); df[DST_IP] = df[DST_IP].astype(str)

sample_class_day = df.groupby([LABEL, "_day"]).size().unstack(fill_value=0)
sample_class_day.to_csv(RES / "sample_class_by_day.csv")
audit = pd.DataFrame({"Item": ["Rows sampled", "Duplicates removed", "Rows used", "Classes", "Days",
                               "Unique src IPs", "Unique dst IPs"],
                      "Value": [n0, n_dup, len(df), df[LABEL].nunique(), df["_day"].nunique(),
                                df[SRC_IP].nunique(), df[DST_IP].nunique()]})
audit.to_csv(RES / "data_quality_audit.csv", index=False)
display(audit); display(sample_class_day)

CLASSES = sorted(df[LABEL].unique())
le = LabelEncoder().fit(CLASSES)
N_CLASSES = len(CLASSES)
BENIGN_ID = int(le.transform(["Benign"])[0])
Y_ALL = le.transform(df[LABEL])

## 6. Label hierarchy (L1 / L2 / L3 / L3-merged)
The model is always trained on the **15 original classes**. Coarser levels are computed from the same predictions.

In [ ]:
def family(name):
    n = name.lower()
    if n == "benign": return "Benign"
    if "ddos" in n: return "DDoS"
    if "dos" in n: return "DoS"
    if any(s in n for s in ["xss", "sql", "brute_force_-web", "web"]): return "Web"
    if "brute" in n: return "BruteForce"
    if "bot" in n: return "Bot"
    if "infil" in n: return "Infiltration"
    return "Other"

L2_NAMES = sorted({family(c) for c in CLASSES})
MAP_L2 = np.array([L2_NAMES.index(family(c)) for c in CLASSES])
L3M_NAMES = sorted({MERGE_GROUPS.get(c, c) for c in CLASSES})
MAP_L3M = np.array([L3M_NAMES.index(MERGE_GROUPS.get(c, c)) for c in CLASSES])
display(pd.DataFrame({"L3 class": CLASSES, "L2 family": [family(c) for c in CLASSES],
                      "L3-merged": [MERGE_GROUPS.get(c, c) for c in CLASSES]}))

## 8. The three splits
* **random** — stratified by (class, day). Comparable with the literature; hosts are shared between train and test.
* **host_disjoint** — every source IP belongs to exactly one split. Classes generated by fewer than 3 source IPs
  (or whose IPs cannot be balanced) **cannot** be host-disjoint; for those classes the rows are split in time blocks
  instead, and the report says so explicitly.
* **temporal** — inside every class: first 70% of its flows (by time) → train, next 15% → val, last 15% → test.

After splitting: train Benign is capped to a balanced size; Val/Test keep `EVAL_BENIGN_RATIO` Benign.

In [ ]:
def _safe_strat(labels, fallback):
    vc = labels.value_counts()
    return labels.where(labels.map(vc) >= 10, fallback)

def split_random():
    strat = _safe_strat(df[LABEL] + "|" + df["_day"], df[LABEL])
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=VAL_FRAC + TEST_FRAC, random_state=SPLIT_SEED, stratify=strat)
    s2 = _safe_strat(strat.iloc[tmp].reset_index(drop=True), df[LABEL].iloc[tmp].reset_index(drop=True))
    va, te = train_test_split(tmp, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), random_state=SPLIT_SEED, stratify=s2)
    rep = pd.DataFrame({"class": CLASSES, "mode": "random (class×day stratified)"})
    return tr, va, te, rep

def _time_blocks(ix):
    ix = df.loc[ix].sort_values("_ts", kind="mergesort").index.values
    a, b = int(len(ix) * TRAIN_FRAC), int(len(ix) * (TRAIN_FRAC + VAL_FRAC))
    return ix[:a], ix[a:b], ix[b:]

def split_temporal():
    parts = [_time_blocks(g.index.values) for _, g in df.groupby(LABEL)]
    tr, va, te = (np.concatenate([p[i] for p in parts]) for i in range(3))
    rep = pd.DataFrame({"class": CLASSES, "mode": "temporal blocks per class"})
    return tr, va, te, rep

def split_host_disjoint():
    rng = np.random.default_rng(SPLIT_SEED)
    target = np.array([TRAIN_FRAC, VAL_FRAC, TEST_FRAC])
    assign = {}
    for c in df.groupby(LABEL)[SRC_IP].nunique().sort_values().index:       # fewest IPs first
        ipc = df.loc[df[LABEL] == c, SRC_IP].value_counts()
        if len(ipc) < 3:
            continue
        load = np.zeros(3)
        for ip, n in ipc.items():
            if ip in assign:
                load[assign[ip]] += n
        free = [ip for ip in ipc.index if ip not in assign]
        rng.shuffle(free)
        for ip in sorted(free, key=lambda x: -ipc[x]):                       # big IPs first, then fill deficits
            s = int(np.argmax(target * ipc.sum() - load)); assign[ip] = s; load[s] += ipc[ip]
    row_split = df[SRC_IP].map(assign)
    out, rep = ([], [], []), []
    for c, g in df.groupby(LABEL):
        s = row_split.loc[g.index]
        shares = np.array([(s == k).mean() for k in range(3)])
        n_ip = g[SRC_IP].nunique()
        if s.isna().any() or shares[0] < 0.3 or shares[1] < 0.05 or shares[2] < 0.05:
            blocks, mode = _time_blocks(g.index.values), f"FALLBACK time blocks ({n_ip} src IPs) — host overlap allowed"
        else:
            blocks, mode = tuple(g.index.values[(s == k).values] for k in range(3)), f"host-disjoint ({n_ip} src IPs)"
        for k in range(3):
            out[k].append(blocks[k])
        rep.append({"class": c, "mode": mode, "src_ips": n_ip})
    tr, va, te = (np.concatenate(o) for o in out)
    return tr, va, te, pd.DataFrame(rep)

SPLITTERS = {"random": split_random, "host_disjoint": split_host_disjoint, "temporal": split_temporal}

def rebalance(tr, va, te):
    rng = np.random.default_rng(SPLIT_SEED)
    def cap_benign(ix, n_keep):
        ix = np.asarray(ix); b = ix[Y_ALL[ix] == BENIGN_ID]; a = ix[Y_ALL[ix] != BENIGN_ID]
        if len(b) > n_keep:
            b = rng.choice(b, n_keep, replace=False)
        return np.sort(np.concatenate([a, b]))          # df is time-sorted -> sorted index = chronological
    tr = cap_benign(tr, int(MAX_PER_CLASS * TRAIN_FRAC))
    ev = lambda ix: cap_benign(ix, int((Y_ALL[ix] != BENIGN_ID).sum() * EVAL_BENIGN_RATIO / (1 - EVAL_BENIGN_RATIO)))
    return tr, ev(va), ev(te)

## 9. Train-only preprocessing, host graph, temporal windows

In [ ]:
def to_num(part, cols):
    X = part[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return np.sign(X) * np.log1p(np.abs(X)) if LOG1P_FEATURES else X

def build_graph(part, X):
    src, dst = part[SRC_IP].values, part[DST_IP].values
    hosts, inv = np.unique(np.concatenate([src, dst]), return_inverse=True)
    s, d = inv[:len(src)], inv[len(src):]
    n = len(hosts)
    out_d, in_d = np.bincount(s, minlength=n), np.bincount(d, minlength=n)
    u_out = pd.Series(d).groupby(s).nunique().reindex(range(n), fill_value=0).values
    u_in = pd.Series(s).groupby(d).nunique().reindex(range(n), fill_value=0).values
    nx_ = np.log1p(np.stack([out_d, in_d, out_d + in_d, u_out, u_in], 1).astype(np.float32))
    nx_ = (nx_ - nx_.mean(0)) / (nx_.std(0) + 1e-6)
    ei = np.stack([np.concatenate([s, d]), np.concatenate([d, s])])
    return Data(x=torch.tensor(nx_, dtype=torch.float32), edge_index=torch.tensor(ei, dtype=torch.long),
                edge_attr=torch.tensor(np.concatenate([X, X])), flow_src=torch.tensor(s, dtype=torch.long),
                flow_dst=torch.tensor(d, dtype=torch.long), flow_attr=torch.tensor(X)).to(device)

def build_windows(part, k, group):
    """Row i = [k-1 previous flows of the same host ..., flow i]; -1 = left padding. Past only."""
    n = len(part)
    t = (part["_ts"].values.astype("datetime64[ns]").astype(np.int64) / 1e9)
    g = np.zeros(n, dtype=np.int64) if group == "global" else \
        pd.factorize(part[SRC_IP if group == "src" else DST_IP])[0]
    order = np.lexsort((np.arange(n), t, g))
    gs = g[order]
    pos = np.arange(n) - np.searchsorted(gs, gs, side="left")
    seq = np.full((n, k), -1, dtype=np.int64)
    r = np.arange(n)
    for j in range(k):
        lag = k - 1 - j
        ok = pos >= lag
        seq[order[ok], j] = order[r[ok] - lag]
    valid = seq >= 0
    dt = np.zeros((n, k), dtype=np.float32)
    tgt = np.broadcast_to(t[:, None], (n, k))
    dt[valid] = np.log1p(np.maximum(tgt[valid] - t[seq[valid]], 0))
    return seq, dt

def prepare(idx3, features, group=WINDOW_GROUP, k=K):
    parts = {n: df.loc[ix].reset_index(drop=True) for n, ix in zip(["train", "val", "test"], idx3)}
    Xtr = to_num(parts["train"], features)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    S = {"features": features, "imputer": imp, "scaler": sc, "group": group, "k": k}
    for n, part in parts.items():
        X = np.clip(sc.transform(imp.transform(to_num(part, features))), -10, 10).astype(np.float32)
        seq, dt = build_windows(part, k, group)
        S[n] = {"df": part, "n": len(part), "graph": build_graph(part, X), "X": torch.tensor(X, device=device),
                "y": torch.tensor(le.transform(part[LABEL]), dtype=torch.long, device=device),
                "seq": torch.tensor(seq, device=device), "dt": torch.tensor(dt, device=device)}
    return S

## 10. Models — identical building blocks
| model | sees graph | sees time |
|---|---|---|
| `gatv2` | ✔ | ✘ |
| `transformer` | ✘ (raw flow features as tokens) | ✔ |
| `two_stage` | ✔ frozen GATv2 embeddings | ✔ |
| `hybrid` | ✔ GATv2 trained jointly (warm-started, frozen for the first epochs) | ✔ |

In [ ]:
class GATEncoder(nn.Module):
    def __init__(self, node_in, edge_in, dropedge=0.0):
        super().__init__()
        self.dropedge = dropedge
        self.gat1 = GATv2Conv(node_in, GNN_HIDDEN, heads=GNN_HEADS, concat=True, edge_dim=edge_in, dropout=DROPOUT)
        self.gat2 = GATv2Conv(GNN_HIDDEN * GNN_HEADS, GNN_HIDDEN, heads=1, concat=False, edge_dim=edge_in, dropout=DROPOUT)
        self.drop = nn.Dropout(DROPOUT)
        self.edge_proj = nn.Sequential(nn.Linear(2 * GNN_HIDDEN + edge_in, EMB_DIM), nn.ReLU(), nn.Dropout(DROPOUT))

    def node_embed(self, g):
        ei, ea = g.edge_index, g.edge_attr
        if self.training and self.dropedge > 0:
            keep = torch.rand(ei.size(1), device=ei.device) >= self.dropedge
            ei, ea = ei[:, keep], ea[keep]
        x = self.drop(F.elu(self.gat1(g.x, ei, ea)))
        return F.elu(self.gat2(x, ei, ea))

    def flow_embed(self, z, g, idx):
        return self.edge_proj(torch.cat([z[g.flow_src[idx]], z[g.flow_dst[idx]], g.flow_attr[idx]], 1))


class TemporalTransformer(nn.Module):
    def __init__(self, tok_in, k):
        super().__init__()
        self.inp = nn.Linear(tok_in + 1, D_MODEL)                      # +1 = log time-gap channel
        self.pos = nn.Parameter(torch.randn(1, k, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, FF_DIM, DROPOUT, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(D_MODEL)

    def forward(self, tok, dt, pad):
        h = self.inp(torch.cat([tok, dt.unsqueeze(-1)], -1)) + self.pos[:, -tok.size(1):]
        return self.norm(self.enc(h, src_key_padding_mask=pad)[:, -1])


class IDSModel(nn.Module):
    def __init__(self, kind, node_in, edge_in, k, dropedge=0.0):
        super().__init__()
        self.kind = kind
        if kind in ("gatv2", "hybrid"):
            self.gnn = GATEncoder(node_in, edge_in, dropedge)
        if kind in ("transformer", "two_stage", "hybrid"):
            self.temporal = TemporalTransformer(edge_in if kind == "transformer" else EMB_DIM, k)
        rep = {"gatv2": EMB_DIM, "transformer": D_MODEL, "two_stage": D_MODEL, "hybrid": D_MODEL + EMB_DIM}[kind]
        self.head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(rep, N_CLASSES))

    def context(self, P):
        return self.gnn.node_embed(P["graph"]) if hasattr(self, "gnn") else None

    def represent(self, P, idx, ctx):
        if self.kind == "gatv2":
            return self.gnn.flow_embed(ctx, P["graph"], idx)
        seq = P["seq"][idx]; pad = seq < 0; safe = seq.clamp(min=0)
        if self.kind == "transformer":
            tok = P["X"][safe]
        elif self.kind == "two_stage":
            tok = P["emb"][safe]
        else:
            tok = self.gnn.flow_embed(ctx, P["graph"], safe.reshape(-1)).view(*safe.shape, -1)
        tok = tok.masked_fill(pad.unsqueeze(-1), 0.0)
        h = self.temporal(tok, P["dt"][idx], pad)
        return torch.cat([h, tok[:, -1]], 1) if self.kind == "hybrid" else h

    def forward(self, P, idx, ctx):
        return self.head(self.represent(P, idx, ctx))

## 11. Training (validation-only selection) and evaluation (L1 / L2 / L3 / L3-merged, FAR, DR)

In [ ]:
@torch.no_grad()
def predict(model, P, want_repr=False, bs=4096):
    model.eval()
    ctx = model.context(P)
    L, R = [], []
    for b in range(0, P["n"], bs):
        idx = torch.arange(b, min(b + bs, P["n"]), device=device)
        r = model.represent(P, idx, ctx)
        L.append(model.head(r)); R.append(r)
    return (torch.cat(L), torch.cat(R)) if want_repr else torch.cat(L)

def train_model(kind, S, seed, init_gnn=None, dropedge=0.0):
    set_seed(seed)
    tr = S["train"]
    model = IDSModel(kind, tr["graph"].x.shape[1], tr["X"].shape[1], S["k"], dropedge).to(device)
    groups = [{"params": [p for n, p in model.named_parameters() if not n.startswith("gnn.")], "lr": LR}]
    if hasattr(model, "gnn"):
        if init_gnn is not None:
            model.gnn.load_state_dict(init_gnn)
        groups.append({"params": model.gnn.parameters(), "lr": HYBRID_GNN_LR if kind == "hybrid" else LR})
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8)
    w = torch.tensor(compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=tr["y"].cpu().numpy()),
                     dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    freeze = HYBRID_FREEZE_EPOCHS if (kind == "hybrid" and init_gnn is not None) else 0
    best, best_state, best_ep, bad, hist = -1.0, None, 0, 0, []
    yv = S["val"]["y"].cpu().numpy()
    for ep in range(1, MAX_EPOCHS + 1):
        if hasattr(model, "gnn"):
            for p in model.gnn.parameters():
                p.requires_grad_(ep > freeze)
        model.train()
        if ep <= freeze:
            model.gnn.eval()
        perm = torch.randperm(tr["n"], device=device); tot = 0.0
        for b in range(0, tr["n"], BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            loss = crit(model(tr, idx, model.context(tr)), tr["y"][idx])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item() * len(idx)
        vl = predict(model, S["val"])
        vf1 = f1_score(yv, vl.argmax(1).cpu().numpy(), average="macro", zero_division=0)
        sched.step(vf1)
        hist.append({"epoch": ep, "train_loss": tot / tr["n"], "val_loss": crit(vl, S["val"]["y"]).item(), "val_macro_f1": vf1})
        if vf1 > best + 1e-5:
            best, best_ep, bad = vf1, ep, 0
            best_state = {k_: v.detach().clone() for k_, v in model.state_dict().items()}
        else:
            bad += 1
        if ep >= MIN_EPOCHS and bad >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return model, {"best_epoch": best_ep, "stopped_epoch": ep, "best_val_macro_f1": best}, pd.DataFrame(hist)

def attack_threshold(val_logits, yv):
    score = 1 - torch.softmax(val_logits, 1)[:, BENIGN_ID].cpu().numpy()
    b = score[yv == BENIGN_ID]
    return float(np.quantile(b, 1 - TARGET_FAR)) if len(b) else 0.5

def evaluate(test_logits, yt, thr):
    prob = torch.softmax(test_logits, 1).cpu().numpy(); p = prob.argmax(1)
    score = 1 - prob[:, BENIGN_ID]
    is_att = yt != BENIGN_ID
    m = {"L3 Macro-F1": f1_score(yt, p, average="macro", zero_division=0),
         "L3 MCC": matthews_corrcoef(yt, p), "L3 Balanced Acc": balanced_accuracy_score(yt, p),
         "L3-merged Macro-F1": f1_score(MAP_L3M[yt], MAP_L3M[p], average="macro", zero_division=0),
         "L2 Macro-F1": f1_score(MAP_L2[yt], MAP_L2[p], average="macro", zero_division=0),
         "L1 FAR (argmax)": float((p[~is_att] != BENIGN_ID).mean()) if (~is_att).any() else np.nan,
         "L1 DR (argmax)": float((p[is_att] != BENIGN_ID).mean()),
         f"L1 FAR @val-thr": float((score[~is_att] > thr).mean()) if (~is_att).any() else np.nan,
         f"L1 DR @val-thr": float((score[is_att] > thr).mean()),
         "L1 ROC-AUC": roc_auc_score(is_att, score) if 0 < is_att.mean() < 1 else np.nan}
    pc = f1_score(yt, p, labels=np.arange(N_CLASSES), average=None, zero_division=0)
    return m, pc, p, score

def per_class_val_loss(val_logits, yv):
    l = F.cross_entropy(val_logits, torch.tensor(yv, device=val_logits.device), reduction="none").cpu().numpy()
    return pd.Series(l).groupby(yv).mean().reindex(range(N_CLASSES)).values

RESULTS, PERCLASS, VALLOSS, SPLIT_REPORTS = [], [], [], []
safe = lambda t: re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

def run_and_log(kind, S, seed, split, tag=None, init_gnn=None, extra=None, dropedge=0.0):
    tag = tag or kind
    t0 = time.time()
    model, info, hist = train_model(kind, S, seed, init_gnn, dropedge)
    vl, tl = predict(model, S["val"]), predict(model, S["test"])
    yv, yt = S["val"]["y"].cpu().numpy(), S["test"]["y"].cpu().numpy()
    thr = attack_threshold(vl, yv)
    m, pc, p, score = evaluate(tl, yt, thr)
    row = {"split": split, "model": tag, "seed": seed, **m, **info, "seconds": round(time.time() - t0), **(extra or {})}
    RESULTS.append(row)
    PERCLASS.append({"split": split, "model": tag, "seed": seed, **dict(zip(CLASSES, pc))})
    VALLOSS.append({"split": split, "model": tag, "seed": seed, **dict(zip(CLASSES, per_class_val_loss(vl, yv)))})
    np.savez_compressed(PRED / f"{split}__{safe(tag)}__seed{seed}.npz", y=yt, pred=p, score=score)
    hist.to_csv(RES / f"history__{split}__{safe(tag)}__seed{seed}.csv", index=False)
    pd.DataFrame(RESULTS).to_csv(RES / "all_runs.csv", index=False)               # saved after every run
    pd.DataFrame(PERCLASS).to_csv(RES / "per_class_f1_all_runs.csv", index=False)
    pd.DataFrame(VALLOSS).to_csv(RES / "per_class_val_loss_all_runs.csv", index=False)
    print(f"  {split:13s} {tag:28s} seed={seed}  L3-F1={m['L3 Macro-F1']:.4f}  L3m-F1={m['L3-merged Macro-F1']:.4f}  "
          f"FAR@thr={m['L1 FAR @val-thr']:.4f}  DR@thr={m['L1 DR @val-thr']:.4f}  "
          f"(best ep {info['best_epoch']}/{info['stopped_epoch']}, {row['seconds']}s)")
    return model

## 12. Robustness to graph change, and the final-model package
* **Graded rewiring:** a fraction *f* of test flows get a random destination host (shuffled among those flows).
  Flow features stay identical, so a model that relies on behaviour should degrade gracefully; a model that relies only on
  "who talks to whom" collapses. The Transformer-only model does not use the graph, so it is the flat reference line.
* **Package:** GATv2 + Two-stage weights, imputer/scaler, feature list, classes and the validation threshold for 1% FAR.
  It is reloaded into fresh models and must reproduce the same test predictions.

In [ ]:
import joblib

def rewire_fraction(S, frac, seed):
    R = dict(S); P = dict(S["test"]); part = P["df"].copy()
    if frac > 0:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(part), int(round(frac * len(part))), replace=False)
        vals = part[DST_IP].values.copy(); vals[idx] = rng.permutation(vals[idx]); part[DST_IP] = vals
    P["df"] = part
    P["graph"] = build_graph(part, P["X"].cpu().numpy())
    if S["group"] != "src":
        seq, dt = build_windows(part, S["k"], S["group"])
        P["seq"], P["dt"] = torch.tensor(seq, device=device), torch.tensor(dt, device=device)
    R["test"] = P
    return R

ROBUST = []

def robustness(gat, two, trf, seed, label):
    yv = S["val"]["y"].cpu().numpy()
    thr_g = attack_threshold(predict(gat, S["val"]), yv)
    thr_t = attack_threshold(predict(two, S["val"]), yv)
    ref = None
    if trf is not None:
        m, *_ = evaluate(predict(trf, S["test"]), S["test"]["y"].cpu().numpy(),
                         attack_threshold(predict(trf, S["val"]), yv))
        ref = m
    for f in REWIRE_FRACTIONS:
        R = rewire_fraction(S, f, seed)
        yt = R["test"]["y"].cpu().numpy()
        mg, *_ = evaluate(predict(gat, R["test"]), yt, thr_g)
        R["test"]["emb"] = predict(gat, R["test"], want_repr=True)[1]
        mt, *_ = evaluate(predict(two, R["test"]), yt, thr_t)
        for name, m in [(f"GATv2 ({label})", mg), (f"Two-stage ({label})", mt)] + \
                       ([("Transformer-only (no graph)", ref)] if ref is not None else []):
            ROBUST.append({"seed": seed, "model": name, "rewired_fraction": f, **m})
        del R
    pd.DataFrame(ROBUST).to_csv(RES / "robustness_rewiring.csv", index=False)
    print(f"  robustness [{label}] seed={seed}: two-stage L3-F1 at 0/30/100% rewired = " +
          " / ".join(f"{r['L3 Macro-F1']:.3f}" for r in ROBUST if r['seed'] == seed and r['model'] == f'Two-stage ({label})'
                     and r['rewired_fraction'] in (0.0, 0.3, 1.0)))

def save_package(gat, two, S, split, seed):
    yv = S["val"]["y"].cpu().numpy()
    thr = attack_threshold(predict(two, S["val"]), yv)
    pkg = {"gatv2_state": {k_: v.cpu() for k_, v in gat.state_dict().items()},
           "two_stage_state": {k_: v.cpu() for k_, v in two.state_dict().items()},
           "dims": {"node_in": int(S["train"]["graph"].x.shape[1]), "edge_in": int(S["train"]["X"].shape[1]), "k": S["k"]},
           "hparams": {"GNN_HIDDEN": GNN_HIDDEN, "GNN_HEADS": GNN_HEADS, "EMB_DIM": EMB_DIM, "DROPOUT": DROPOUT,
                       "D_MODEL": D_MODEL, "N_HEADS": N_HEADS, "N_LAYERS": N_LAYERS, "FF_DIM": FF_DIM, "DROPEDGE": DROPEDGE},
           "window_group": S["group"], "classes": CLASSES, "benign_id": BENIGN_ID, "features": MODEL_FEATURES,
           "log1p": LOG1P_FEATURES, "attack_threshold": thr, "target_far": TARGET_FAR,
           "merge_groups": MERGE_GROUPS, "trained_on": {"split": split, "seed": seed}}
    torch.save(pkg, MOD / "final_model_package.pt")
    joblib.dump({"imputer": S["imputer"], "scaler": S["scaler"], "label_encoder": le, "features": MODEL_FEATURES,
                 "log1p": LOG1P_FEATURES}, MOD / "final_preprocessing.joblib")
    # ---- reload check: fresh models must give identical test predictions ----
    p = torch.load(MOD / "final_model_package.pt", map_location=device)
    g2 = IDSModel("gatv2", p["dims"]["node_in"], p["dims"]["edge_in"], p["dims"]["k"]).to(device)
    g2.load_state_dict(p["gatv2_state"])
    t2 = IDSModel("two_stage", p["dims"]["node_in"], p["dims"]["edge_in"], p["dims"]["k"]).to(device)
    t2.load_state_dict(p["two_stage_state"])
    T = dict(S["test"]); T["emb"] = predict(g2, T, want_repr=True)[1]
    same = (predict(t2, T).argmax(1) == predict(two, S["test"]).argmax(1)).float().mean().item()
    assert same > 0.999, f"package reload mismatch ({same:.4f})"
    print(f"  final model package saved and reload-verified (agreement {same:.4f}); attack threshold = {thr:.4f}")

## 13. Main experiment — every split × every seed (same split for every model)

In [ ]:
def set_embeddings(gat, S):
    for n in ["train", "val", "test"]:
        S[n]["emb"] = predict(gat, S[n], want_repr=True)[1]

MAIN_MODELS = ["gatv2", "transformer", "two_stage"] + (["hybrid"] if RUN_HYBRID else [])

for split in SPLITS_TO_RUN:
    print(f"\n==================== SPLIT: {split} ====================")
    tr, va, te, rep = SPLITTERS[split]()
    tr, va, te = rebalance(tr, va, te)
    sizes = pd.DataFrame({n: pd.Series(Y_ALL[ix]).map(dict(enumerate(CLASSES))).value_counts()
                          for n, ix in [("train", tr), ("val", va), ("test", te)]}).fillna(0).astype(int)
    rep.set_index("class").join(sizes).to_csv(RES / f"split_report__{split}.csv")
    SPLIT_REPORTS.append({"split": split, "test_rows_whose_src_ip_is_in_train": float(df.loc[te, SRC_IP].isin(set(df.loc[tr, SRC_IP])).mean()),
                          "benign_share_val": float((Y_ALL[va] == BENIGN_ID).mean()),
                          "benign_share_test": float((Y_ALL[te] == BENIGN_ID).mean())})
    print(pd.DataFrame([SPLIT_REPORTS[-1]]).round(3).to_string(index=False))

    S = prepare((tr, va, te), MODEL_FEATURES)
    for si, seed in enumerate(SEEDS):
        print(f"  --- seed {seed} ---")
        gat = run_and_log("gatv2", S, seed, split, dropedge=DROPEDGE)
        set_embeddings(gat, S)
        two = run_and_log("two_stage", S, seed, split)
        trf = run_and_log("transformer", S, seed, split)
        if RUN_HYBRID:
            run_and_log("hybrid", S, seed, split, dropedge=DROPEDGE,
                        init_gnn={k_[4:]: v.clone() for k_, v in gat.state_dict().items() if k_.startswith("gnn.")})
        if split == PACKAGE_SPLIT and si == 0:
            save_package(gat, two, S, split, seed)
        if split == "random" and si < N_ROBUST_SEEDS:
            robustness(gat, two, trf, seed, f"DropEdge {DROPEDGE}")
            if RUN_DROPEDGE_CONTROL:
                g0 = run_and_log("gatv2", S, seed, split, tag="control: GATv2 no DropEdge", dropedge=0.0)
                set_embeddings(g0, S)
                t0 = run_and_log("two_stage", S, seed, split, tag="control: Two-stage no DropEdge")
                robustness(g0, t0, None, seed, "no DropEdge")
                del g0, t0
        del gat, two, trf; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    del S; gc.collect()

pd.DataFrame(SPLIT_REPORTS).to_csv(RES / "split_overview.csv", index=False)

## 14. Summary — mean ± std over seeds

In [ ]:
res = pd.DataFrame(RESULTS)
KEY = ["L3 Macro-F1", "L3-merged Macro-F1", "L2 Macro-F1", "L3 MCC", "L1 FAR @val-thr", "L1 DR @val-thr",
       "L1 FAR (argmax)", "L1 DR (argmax)", "L1 ROC-AUC", "best_epoch"]
summary = res.groupby(["split", "model"], sort=False)[KEY].agg(["mean", "std"]).round(4)
summary.to_csv(RES / "summary_mean_std.csv")
display(summary)

rob = pd.DataFrame(ROBUST)
if len(rob):
    rob_s = rob.groupby(["model", "rewired_fraction"])[["L3 Macro-F1", "L3-merged Macro-F1", "L1 DR @val-thr", "L1 FAR @val-thr"]].mean().round(4)
    rob_s.to_csv(RES / "robustness_summary.csv"); display(rob_s.unstack(0)["L3 Macro-F1"])

## 15. Statistics — paired over the seeds both models share
Verdict = "A better" only if the paired t-test p < 0.05 **and** the bootstrap CI of the first common seed excludes 0.

In [ ]:
def load_pred(split, model, seed):
    z = np.load(PRED / f"{split}__{safe(model)}__seed{seed}.npz"); return z["y"], z["pred"]

def compare(split, a, b, metric="L3 Macro-F1"):
    A = res[(res.split == split) & (res.model == a)].set_index("seed")[metric]
    B = res[(res.split == split) & (res.model == b)].set_index("seed")[metric]
    seeds = sorted(set(A.index) & set(B.index))
    if len(seeds) < 2:
        return None
    A, B = A.loc[seeds].values, B.loc[seeds].values; d = A - B
    p_t = ttest_rel(A, B).pvalue
    p_w = wilcoxon(A, B).pvalue if np.any(d != 0) else np.nan
    mc = []
    for s in seeds:
        y, pa = load_pred(split, a, s); _, pb = load_pred(split, b, s)
        n01, n10 = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
        mc.append(binomtest(min(n01, n10), n01 + n10).pvalue if n01 + n10 else 1.0)
    y, pa = load_pred(split, a, seeds[0]); _, pb = load_pred(split, b, seeds[0])
    rs = np.random.default_rng(0); boots = []
    for _ in range(N_BOOTSTRAP):
        i = rs.integers(0, len(y), len(y))
        boots.append(f1_score(y[i], pa[i], average="macro", zero_division=0) - f1_score(y[i], pb[i], average="macro", zero_division=0))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    verdict = "A better" if (d.mean() > 0 and p_t < 0.05 and lo > 0) else \
              "B better" if (d.mean() < 0 and p_t < 0.05 and hi < 0) else "no significant difference"
    return {"split": split, "metric": metric, "A": a, "B": b, "n_seeds": len(seeds), "mean diff (A-B)": d.mean(),
            "std diff": d.std(ddof=1), "A wins": f"{int((d > 0).sum())}/{len(d)}", "paired t p": p_t, "Wilcoxon p": p_w,
            "McNemar p<0.05": f"{sum(p < 0.05 for p in mc)}/{len(mc)}", "bootstrap 95% CI (1st seed)": f"[{lo:+.4f}, {hi:+.4f}]",
            "verdict": verdict}

PAIRS = [("two_stage", "transformer"), ("two_stage", "gatv2"), ("transformer", "gatv2")]
if RUN_HYBRID:
    PAIRS += [("hybrid", "two_stage"), ("hybrid", "transformer")]
rows = [compare(s, a, b, m) for s in SPLITS_TO_RUN for a, b in PAIRS for m in ["L3 Macro-F1", "L3-merged Macro-F1"]]
if RUN_DROPEDGE_CONTROL:
    rows += [compare("random", "two_stage", "control: Two-stage no DropEdge", "L3 Macro-F1"),
             compare("random", "gatv2", "control: GATv2 no DropEdge", "L3 Macro-F1")]
stats = pd.DataFrame([r for r in rows if r])
stats.to_csv(RES / "statistical_tests.csv", index=False)
display(stats.round(4))

## 16. Figures

In [ ]:
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
COL = {"gatv2": "#8fa9bf", "transformer": "#c9a66b", "two_stage": "#2e5e8c", "hybrid": "#6b9c8a"}
main = res[res.model.isin(MAIN_MODELS)]
mods = [m for m in MAIN_MODELS if m in main.model.unique()]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, met in zip(axes, ["L3 Macro-F1", "L3-merged Macro-F1", "L1 DR @val-thr"]):
    x = np.arange(len(SPLITS_TO_RUN)); w = 0.8 / len(mods)
    for j, m in enumerate(mods):
        g = main[main.model == m].groupby("split")[met].agg(["mean", "std"]).reindex(SPLITS_TO_RUN)
        ax.bar(x + j * w - 0.4 + w / 2, g["mean"], w, yerr=g["std"].fillna(0), color=COL[m], label=m, capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(SPLITS_TO_RUN); ax.set_title(met)
    ax.set_ylim(max(0, np.nanmin(main[met]) - 0.1), 1.0); ax.grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False)
plt.suptitle(f"Mean ± std over {len(SEEDS)} seeds (DR at the threshold giving {TARGET_FAR:.0%} FAR on validation)", y=1.03)
plt.savefig(FIG / "01_models_by_split.png", dpi=160, bbox_inches="tight"); plt.show()

if len(rob):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    styles = {"Transformer-only (no graph)": ("#c9a66b", "-", "o")}
    for ax, met in zip(axes, ["L3 Macro-F1", "L1 DR @val-thr"]):
        for name, g in rob.groupby("model"):
            m_ = g.groupby("rewired_fraction")[met].agg(["mean", "std"])
            c = "#c9a66b" if "Transformer" in name else ("#2e5e8c" if "Two-stage" in name else "#8fa9bf")
            ls = "--" if "no DropEdge" in name else "-"
            ax.errorbar(m_.index * 100, m_["mean"], yerr=m_["std"].fillna(0), color=c, ls=ls, marker="o", capsize=3, label=name)
        ax.set_xlabel("% of test flows with a random destination host"); ax.set_title(met); ax.grid(alpha=0.25)
    axes[0].legend(frameon=False, fontsize=8)
    plt.suptitle(f"Robustness to graph change — random split, {N_ROBUST_SEEDS} seeds (dashed = no DropEdge)", y=1.03)
    plt.savefig(FIG / "02_robustness_rewiring.png", dpi=160, bbox_inches="tight"); plt.show()

pcf = pd.DataFrame(PERCLASS); pcf = pcf[pcf.model.isin(mods)]
for split in SPLITS_TO_RUN:
    Mx = pcf[pcf.split == split].groupby("model")[CLASSES].mean().reindex(mods).T
    fig, ax = plt.subplots(figsize=(1.4 * len(mods) + 4, 0.42 * N_CLASSES + 1.5))
    ax.imshow(Mx.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(mods))); ax.set_xticklabels(mods); ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES)
    for i in range(N_CLASSES):
        for j in range(len(mods)):
            ax.text(j, i, f"{Mx.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if Mx.values[i, j] > 0.6 else "black")
    ax.set_title(f"Per-class F1 — {split} split (mean over seeds)")
    plt.savefig(FIG / f"03_per_class_f1__{split}.png", dpi=160, bbox_inches="tight"); plt.show()

for split in SPLITS_TO_RUN:
    f = PRED / f"{split}__two_stage__seed{SEEDS[0]}.npz"
    if not f.exists():
        continue
    z = np.load(f); cm = confusion_matrix(z["y"], z["pred"], labels=np.arange(N_CLASSES)).astype(float)
    cm /= np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(11, 9)); ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASSES, rotation=75, ha="right", fontsize=8)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASSES, fontsize=8)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            if cm[i, j] >= 0.01:
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if cm[i, j] > 0.6 else "black")
    ax.set_title(f"Two-stage — row-normalised confusion matrix — {split} split"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.savefig(FIG / f"04_confusion_two_stage__{split}.png", dpi=160, bbox_inches="tight"); plt.show()

## 17. Package everything

In [ ]:
zip_path = shutil.make_archive(str(WORK_DIR.parent / "FINAL_V3_RESULTS"), "zip", root_dir=str(WORK_DIR))
print("RUN COMPLETE ->", zip_path)
print("Final deployable model:", MOD / "final_model_package.pt", "+", MOD / "final_preprocessing.joblib")

## How to read the results
1. **statistical_tests.csv** — with 10 seeds: is Two-stage significantly better than Transformer-only and GATv2 on each split?
2. **robustness_summary.csv / 02_robustness_rewiring.png** — how fast each model degrades when the graph changes, and whether DropEdge helps.
   Also check the `control:` rows: DropEdge must not cost accuracy at 0% rewiring.
3. **split_overview.csv** — Benign share in Val/Test should now be ≈ 80%.
4. **final_model_package.pt + final_preprocessing.joblib** — the locked model for explainability, real-time and the dashboard.